In [1]:
import xml.etree.ElementTree as ET

def slot_to_time(slot_index):
    # Based on 288 slots per day, 1 slot = 5 minutes
    total_minutes = int(slot_index) * 5
    hours = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours:02d}:{minutes:02d}"

def decode_days(day_bits):
    days_map = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    # day_bits is a string like '1000000'
    active_days = [days_map[i] for i, bit in enumerate(day_bits) if bit == '1']
    return ", ".join(active_days)

def parse_schedule(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    print(f"{'Course':<10} | {'Class':<10} | {'Days':<15} | {'Start':<10} | {'Duration'}")
    print("-" * 65)

    for course in root.findall('.//course'):
        course_id = course.get('id')
        for config in course.findall('config'):
            for subpart in config.findall('subpart'):
                for cls in subpart.findall('class'):
                    class_id = cls.get('id')
                    # A class might have multiple suggested times; usually we pick the first or valid one
                    time_elem = cls.find('time')
                    if time_elem is not None:
                        days = decode_days(time_elem.get('days'))
                        start = slot_to_time(time_elem.get('start'))
                        duration = int(time_elem.get('length')) * 5 # in minutes
                        
                        print(f"{course_id:<10} | {class_id:<10} | {days:<15} | {start:<10} | {duration} mins")

# Run the function
parse_schedule('/home/emery/Documents/Scheduling_Puma_Optimizer/notebooks/agh-fis-spr17.xml')

Course     | Class      | Days            | Start      | Duration
-----------------------------------------------------------------
1          | 1          | Mon             | 07:45      | 90 mins
1          | 2          | Mon             | 07:45      | 135 mins
1          | 3          | Mon             | 07:45      | 135 mins
1          | 4          | Mon             | 07:45      | 135 mins
1          | 5          | Mon             | 07:45      | 135 mins
1          | 6          | Mon             | 07:45      | 135 mins
1          | 7          | Mon             | 07:45      | 135 mins
1          | 8          | Mon             | 07:45      | 135 mins
1          | 9          | Mon             | 07:45      | 135 mins
2          | 10         | Tue             | 07:45      | 90 mins
2          | 11         | Fri             | 07:45      | 135 mins
2          | 12         | Mon             | 07:45      | 135 mins
2          | 13         | Mon             | 07:45      | 135 mins
2          |

In [2]:
import xml.etree.ElementTree as ET

def get_room_availability(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    nr_days = int(root.get('nrDays'))
    slots_per_day = int(root.get('slotsPerDay'))
    nr_weeks = int(root.get('nrWeeks'))
    
    rooms_data = {}
    
    for room in root.findall('.//room'):
        room_id = room.get('id')
        # If unavailable attribute is missing, it's always available (all '0')
        unavailable = room.get('unavailable', '0' * (nr_days * slots_per_day * nr_weeks))
        rooms_data[room_id] = unavailable

    return rooms_data, nr_days, slots_per_day

def is_room_free(rooms_data, room_id, week, day, slot, nr_days, slots_per_day):
    # Calculate linear index
    index = (week * nr_days * slots_per_day) + (day * slots_per_day) + slot
    
    bitstring = rooms_data.get(room_id)
    if not bitstring:
        return True # Room not restricted
        
    # '0' means available, '1' means unavailable
    return bitstring[index] == '0'

# Execution
data, days, slots = get_room_availability('agh-fis-spr17.xml')

# Example: Is Room 1 free on Week 0, Day 0 (Monday), at Slot 96 (08:00 AM)?
free = is_room_free(data, '1', week=0, day=0, slot=96, nr_days=days, slots_per_day=slots)
print(f"Room 1 availability at 08:00 AM Monday: {'Available' if free else 'Occupied'}")

Room 1 availability at 08:00 AM Monday: Available


In [1]:
import xml.etree.ElementTree as ET

class TimetableSolver:
    def __init__(self, file_path):
        self.tree = ET.parse(file_path)
        self.root = self.tree.getroot()
        self.nr_days = int(self.root.get('nrDays', 7))
        self.rooms = self._parse_rooms() # Fixed parsing
        self.classes = self._parse_classes()

    def _parse_rooms(self):
        rooms = {}
        # CORRECTED: Only fetch room definitions from the <rooms> block
        rooms_section = self.root.find('rooms')
        if rooms_section is not None:
            for r in rooms_section.findall('room'):
                rooms[r.get('id')] = {
                    'capacity': int(r.get('capacity')), # Should not error now
                    'unavailable': r.get('unavailable', '')
                }
        return rooms

    def _parse_classes(self):
        classes = {}
        # Use recursive search for classes as they are nested in config/subpart
        for cls in self.root.findall('.//class'):
            classes[cls.get('id')] = {
                'limit': int(cls.get('limit', '0')),
                # Parse candidate rooms
                'rooms': [r.get('id') for r in cls.findall('room')],
                # Parse candidate times
                'times': [{'days': t.get('days'), 
                           'start': int(t.get('start')), 
                           'length': int(t.get('length'))} 
                          for t in cls.findall('time')]
            }
        return classes

    def solve_greedy(self):
        schedule = {}
        unscheduled = []
        
        # Sort by difficulty (classes with fewer room options go first)
        sorted_classes = sorted(self.classes.items(), key=lambda x: len(x[1]['rooms']))

        for class_id, reqs in sorted_classes:
            assigned = False
            
            # 1. Try Rooms
            for room_id in reqs['rooms']:
                if room_id not in self.rooms: continue
                
                # Capacity Check (Now works because capacity is not 0)
                if reqs['limit'] > self.rooms[room_id]['capacity']: continue

                # 2. Try Times
                for time in reqs['times']:
                    if self._is_slot_free(schedule, room_id, time):
                        schedule[class_id] = {'room': room_id, 'time': time}
                        assigned = True
                        break 
                if assigned: break
            
            if not assigned:
                unscheduled.append(class_id)
                
        return schedule, unscheduled

    def _is_slot_free(self, schedule, room_id, new_time):
        for assigned in schedule.values():
            if assigned['room'] == room_id:
                # Check for time overlap
                # (Simple check: exact slot match. Real logic needs to check duration overlap)
                t1_start = assigned['time']['start']
                t1_end = t1_start + assigned['time']['length']
                t2_start = new_time['start']
                t2_end = t2_start + new_time['length']
                
                # Check day overlap (bitwise string match)
                days_overlap = False
                for i in range(7):
                    if assigned['time']['days'][i] == '1' and new_time['days'][i] == '1':
                        days_overlap = True
                        break
                
                if days_overlap:
                    # If days overlap, check if time slots overlap
                    if max(t1_start, t2_start) < min(t1_end, t2_end):
                        return False
        return True

    def print_results(self, schedule, failed):
        print(f"Successfully Scheduled: {len(schedule)}")
        print(f"Unscheduled (Conflicts): {len(failed)}")
        
        print("\nSample Assignments:")
        print(f"{'Class':<8} | {'Room':<5} | {'Day':<5} | {'Time':<5}")
        print("-" * 30)
        for cid, data in list(schedule.items())[:5]:
            # Simple day parser
            d_idx = data['time']['days'].find('1')
            day = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][d_idx]
            
            # Simple time parser
            minutes = data['time']['start'] * 5
            time_str = f"{minutes//60:02d}:{minutes%60:02d}"
            
            print(f"{cid:<8} | {data['room']:<5} | {day:<5} | {time_str:<5}")

# Execute
solver = TimetableSolver('agh-fis-spr17.xml')
schedule, failed = solver.solve_greedy()
solver.print_results(schedule, failed)

Successfully Scheduled: 16
Unscheduled (Conflicts): 1223

Sample Assignments:
Class    | Room  | Day   | Time 
------------------------------
367      | 77    | Fri   | 15:30
579      | 75    | Thu   | 11:00
597      | 40    | Fri   | 14:30
598      | 52    | Fri   | 14:30
601      | 6     | Mon   | 11:20
